Overall Insights:

  Germany’s EV charging infrastructure is not just expanding in numbers, but improving in quality and power output.
  
  The rapid growth of HPCs signals readiness for mass EV adoption and long-distance travel, aligning with sustainability goals.
  
  Market maturity is evident: fewer new stations, but more powerful and strategically placed chargers.
  
  Regional and operator-level drill-downs support targeted decision-making for stakeholders.
  
  The dashboard tells a story of transition—from early expansion to strategic, user-focused development—accelerating sustainable mobility.

 This notebook’s visualizations collectively demonstrate Germany’s leadership in EV infrastructure, highlighting both progress and areas for continued improvement. The data-driven approach supports strategic planning and showcases the country’s commitment to a sustainable transportation future.

In [0]:
# Load the EV charging station data
# Replace the table name below with the correct one if needed
ladestation_df = spark.table("`germany's_ev_charging_infrastructure_status`.default.ladestation_fact_table_1")

# Group by commissioning date and count stations
trend_df = (
    ladestation_df
    .groupBy("Inbetriebnahmedatum")
    .count()
    .orderBy("Inbetriebnahmedatum")
    .withColumnRenamed("count", "number_of_stations")
)

# Display the trend
display(trend_df)

In [0]:
import matplotlib.pyplot as plt

trend_pd = trend_df.toPandas()

plt.figure(figsize=(12, 6))
plt.plot(trend_pd["Inbetriebnahmedatum"], trend_pd["number_of_stations"], marker='o')
plt.xlabel("Inbetriebnahmedatum")
plt.ylabel("Number of Stations")
plt.title("Trend of EV Charging Stations Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
import plotly.express as px

trend_pd = trend_df.toPandas()

fig = px.line(
    trend_pd,
    x="Inbetriebnahmedatum",
    y="number_of_stations",
    markers=True,
    title="Trend of EV Charging Stations Over Time",
    labels={"Inbetriebnahmedatum": "Inbetriebnahmedatum", "number_of_stations": "Number of Stations"}
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

 Trend of EV Charging Stations Over Time:

  The line charts show a consistent increase in the number of charging stations commissioned, with notable growth periods.
  
  This upward trend reflects sustained investment and policy support, making EVs more accessible and reducing range anxiety.
  
  Occasional plateaus or dips may indicate market saturation, regulatory changes, or shifts in funding.


The trend visualizations show how the number of EV charging stations commissioned in Germany has changed over time.
An upward trend indicates increasing deployment of charging infrastructure, supporting EV adoption.
Peaks or dips may correspond to policy changes, funding cycles, or market dynamics.
Consistent growth suggests ongoing investment, while plateaus or declines may signal market saturation or regulatory barriers.
For deeper insights, correlate these trends with external factors such as government incentives, EV sales, or regional policies.

In [0]:
from pyspark.sql.functions import when, col

energy_grouped_df = ladestation_df.withColumn(
    "energy_output_group",
    when(col("NennleistungBNetzA") < 22, "Standard AC chargers (<22kW)")
    .when((col("NennleistungBNetzA") >= 22) & (col("NennleistungBNetzA") <= 150), "Faster DC chargers (23-150kW)")
    .when(col("NennleistungBNetzA") > 150, "HPC (High Power Chargers (>150kW)")
)

grouped_count_df = (
    energy_grouped_df
    .groupBy("energy_output_group")
    .count()
    .orderBy("energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

display(grouped_count_df)

In [0]:
energy_trend_df = (
    energy_grouped_df
    .groupBy("Inbetriebnahmedatum", "energy_output_group")
    .count()
    .orderBy("Inbetriebnahmedatum", "energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

display(energy_trend_df)

In [0]:
import plotly.express as px
from pyspark.sql.functions import year

energy_trend_year_df = (
    energy_grouped_df
    .withColumn("year", year(col("Inbetriebnahmedatum")))
    .groupBy("year", "energy_output_group")
    .count()
    .orderBy("year", "energy_output_group")
    .withColumnRenamed("count", "number_of_stations")
)

energy_trend_year_pd = energy_trend_year_df.toPandas()

fig = px.line(
    energy_trend_year_pd,
    x="year",
    y="number_of_stations",
    color="energy_output_group",
    markers=True,
    title="Growth of Energy Output Charger Over Time (Yearly)",
    labels={"year": "Year", "number_of_stations": "Number of Stations", "energy_output_group": "Energy Output Group"}
)
fig.update_layout(xaxis_tickangle=0)
fig.show()

Charging Capacity Mix:

  The grouped bar and line charts reveal a clear increase in all charger types, especially High Power Chargers (HPC) in recent years.
  
  The infrastructure is evolving from standard AC (<22kW) to more rapid DC and HPC (>150kW), supporting faster charging and long-distance travel.

  This shift from quantity to quality demonstrates a maturing market focused on user experience and strategic deployment.


In [0]:
from pyspark.sql.functions import col

# Group by operator and count stations
operator_count_df = (
    ladestation_df
    .groupBy("Betreiber")
    .count()
    .orderBy(col("count").desc())
    .limit(10)
    .withColumnRenamed("count", "number_of_stations")
)

display(operator_count_df)


In [0]:
import plotly.express as px

operator_count_pd = operator_count_df.orderBy(col("number_of_stations").desc()).toPandas()

fig = px.bar(
    operator_count_pd,
    x="number_of_stations",
    y="Betreiber",
    orientation="h",
    title="Top 10 Operators by Number of EV Charging Stations",
    labels={"Betreiber": "Operator", "number_of_stations": "Number of Stations"}
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

Operator Market Overview:

  The bar chart highlights the top operators by station count, showing market concentration and competitive dynamics.
  
  A few operators dominate, but there is diversity, suggesting healthy competition and potential for further consolidation or innovation.


In [0]:
from pyspark.sql.functions import col

# Group by Bundesland and count stations
bundesland_count_df = (
    ladestation_df
    .groupBy("Bundesland")
    .count()
    .orderBy(col("count").desc())
    .withColumnRenamed("count", "number_of_stations")
)

display(bundesland_count_df)

In [0]:
import plotly.express as px

bundesland_count_pd = bundesland_count_df.toPandas()
max_idx = bundesland_count_pd["number_of_stations"].idxmax()
colors = ['#636EFA'] * len(bundesland_count_pd)
colors[max_idx] = '#EF553B'  # Unique color for the highest

fig = px.bar(
    bundesland_count_pd,
    x="number_of_stations",
    y="Bundesland",
    orientation="h",
    title="EV Charging Stations by Bundesland",
    labels={"Bundesland": "Bundesland", "number_of_stations": "Number of Stations"}
)
fig.update_traces(marker_color=colors)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

 Regional Distribution:

  The Bundesland bar chart shows regional disparities, with some states leading in station deployment.
  
  This enables analysis for strategic planning, identifying areas needing more investment or policy support.


In [0]:
from pyspark.sql.functions import sum

# Total number of stations
total_stations = ladestation_df.count()

# Total number of charging points
total_charging_points = ladestation_df.agg(sum("AnzahlLadepunkteBNetzA")).collect()[0][0]

kpi_df = spark.createDataFrame(
    [
        ("Total Stations", total_stations),
        ("Total Charging Points", total_charging_points)
    ],
    ["KPI", "Value"]
)

display(kpi_df)

 Key Performance Indicators:

  KPIs summarize the total number of stations and charging points, providing a snapshot of infrastructure scale and growth.
